# Domain 1: Agentic Architecture & Orchestration
**Exam**: Claude Certified Architect – Foundations (CCAR-F)
**Weighting**: 27% of exam
**Depth**: deep

## Domain Overview

This domain covers designing and implementing agentic loops, orchestrating multi-agent
coordinator/subagent systems, enforcing critical workflow steps deterministically, decomposing
complex tasks, and managing session state across long-running agent work.

**Why it matters**: this is the single largest domain on the exam (27%). Architects are expected to
know not just how to call an LLM, but how to build reliable, multi-step, multi-agent systems that
behave predictably in production.

**Key concepts to know**: the agentic loop (`stop_reason` control flow), coordinator/subagent
patterns with the Task tool, hooks for deterministic enforcement, task decomposition strategies, and
session resumption/forking.

In [ ]:
# Setup: these notebooks demonstrate Claude Agent SDK / Claude API / MCP concepts using
# plain Python simulations of the relevant objects and control flow, so every cell is runnable
# without needing a live Claude Agent SDK installation or API key. Where call_claude() is used,
# it falls back to a simulated response if ANTHROPIC_API_KEY is not set.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception:
        return f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY to call the real API] Prompt: {prompt[:120]}..."

print("call_claude() ready. Concepts below use plain Python to simulate Agent SDK / MCP / Claude Code behavior.")

## Key Concepts

### Design and implement agentic loops for autonomous task execution

The agentic loop lifecycle is: send a request, inspect `stop_reason` (`"tool_use"` means
Claude wants to call a tool; `"end_turn"` means it's done), execute requested tools, append results
to conversation history, and repeat. Anti-patterns include parsing natural-language text to decide
when to stop, or using an arbitrary iteration cap as the *primary* stopping mechanism.

In [ ]:
def agentic_loop(client_call, messages, tools, max_iterations=10):
    """Simplified agentic loop driven by stop_reason, not by parsing text or a hard iteration cap alone."""
    for i in range(max_iterations):
        response = client_call(messages, tools)  # response has .stop_reason and .content
        if response.stop_reason == "end_turn":
            return response
        if response.stop_reason == "tool_use":
            tool_results = [execute_tool(block) for block in response.content if block.type == "tool_use"]
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})  # results feed back into context
        else:
            raise RuntimeError(f"Unhandled stop_reason: {response.stop_reason}")
    raise RuntimeError("Exceeded max_iterations without end_turn - investigate, don't just raise the cap.")

def execute_tool(tool_use_block):
    return {"type": "tool_result", "tool_use_id": tool_use_block.id, "content": "simulated result"}

print("Agentic loop defined: continues on 'tool_use', stops on 'end_turn'.")

### Orchestrate multi-agent systems with coordinator-subagent patterns

A coordinator agent manages all inter-subagent communication (hub-and-spoke), decomposing
tasks and routing them to specialized subagents based on query complexity — rather than always
invoking every subagent regardless of need. Subagents operate with **isolated context**: they do not
automatically inherit the coordinator's conversation history, so relevant findings must be passed
explicitly. Spawning multiple subagents in parallel means emitting multiple Task tool calls in a
single coordinator turn, not across separate turns.

In [ ]:
def coordinator_decide_subagents(query: str) -> list:
    """Dynamically select which subagents to invoke, rather than always routing through the full pipeline."""
    needs_web_search = "current" in query or "latest" in query
    needs_doc_analysis = "document" in query or "report" in query
    subagents = []
    if needs_web_search:
        subagents.append("web_search_agent")
    if needs_doc_analysis:
        subagents.append("document_analysis_agent")
    subagents.append("synthesis_agent")  # always needed to combine findings
    return subagents

print(coordinator_decide_subagents("What are the latest trends in this document?"))
print(coordinator_decide_subagents("Summarize this document."))

### Apply enforcement patterns: hooks vs. prompt-based guidance

Programmatic enforcement (hooks, prerequisite gates) provides deterministic guarantees;
prompt-based instructions alone have a non-zero failure rate and are insufficient when business
rules require guaranteed compliance (e.g., verifying identity before a refund). A `PostToolUse` hook
can normalize heterogeneous tool output formats, or intercept and block a policy-violating tool call
before it executes (e.g., blocking refunds above a threshold).

In [ ]:
def pre_tool_call_hook(tool_name: str, tool_input: dict, verified_customer_id: str | None) -> bool:
    """Programmatic prerequisite gate: blocks process_refund/lookup_order until identity is verified."""
    if tool_name in {"process_refund", "lookup_order"} and not verified_customer_id:
        return False  # block the call - prompt instructions alone would not guarantee this
    if tool_name == "process_refund" and tool_input.get("amount", 0) > 500:
        return False  # policy: refunds over $500 require human escalation
    return True

print(pre_tool_call_hook("process_refund", {"amount": 200}, verified_customer_id=None))   # blocked: no verified ID
print(pre_tool_call_hook("process_refund", {"amount": 200}, verified_customer_id="C123")) # allowed
print(pre_tool_call_hook("process_refund", {"amount": 900}, verified_customer_id="C123")) # blocked: over threshold

### Design task decomposition strategies for complex workflows

Use fixed, sequential **prompt chaining** for predictable multi-aspect reviews (e.g., analyze
each file individually, then a separate cross-file integration pass). Use **dynamic, adaptive
decomposition** for open-ended investigation tasks, where subtasks are generated based on what is
discovered at each step rather than fixed up front.

In [ ]:
def decompose_code_review(files: list[str]) -> list[str]:
    """Prompt chaining: per-file passes plus one integration pass, to avoid attention dilution."""
    per_file_passes = [f"Analyze {f} in isolation for local issues." for f in files]
    integration_pass = "Analyze cross-file data flow and interactions across: " + ", ".join(files)
    return per_file_passes + [integration_pass]

for step in decompose_code_review(["orders.py", "refunds.py", "customers.py"]):
    print("-", step)

### Manage session state, resumption, and forking

`--resume <session-name>` continues a specific prior conversation. `fork_session` creates an
independent branch from a shared analysis baseline to explore divergent approaches (e.g., comparing
two refactoring strategies). When resuming after code changes, inform the agent about exactly what
changed rather than relying on stale tool results, or start fresh with a structured summary if most
prior context is no longer valid.

In [ ]:
def choose_session_strategy(prior_context_valid: bool, need_parallel_exploration: bool) -> str:
    if need_parallel_exploration:
        return "fork_session from the shared baseline to explore branches independently"
    if prior_context_valid:
        return "--resume <session-name> to continue with existing context"
    return "start a new session with an injected structured summary (avoid stale tool results)"

print(choose_session_strategy(prior_context_valid=True, need_parallel_exploration=False))
print(choose_session_strategy(prior_context_valid=False, need_parallel_exploration=False))
print(choose_session_strategy(prior_context_valid=True, need_parallel_exploration=True))

## Mini Hands-On Project

**Covers**: agentic loop design, coordinator-subagent orchestration, enforcement hooks.

**Scenario**: Build a simplified customer support resolution agent (from the exam's Scenario 1) that
verifies identity before processing a refund, using a hook for deterministic enforcement, and
decomposes multi-issue requests.

### Step 1: Define tools and a verification-gate hook

In [ ]:
verified_customers = {"C123": "Jane Doe"}

def get_customer(name: str):
    for cid, cname in verified_customers.items():
        if cname.lower() == name.lower():
            return cid
    return None

def process_refund(customer_id: str, amount: float):
    return f"Refunded ${amount} to {customer_id}"

def pre_tool_call_hook(tool_name, tool_input, verified_customer_id):
    if tool_name == "process_refund" and not verified_customer_id:
        return False
    if tool_name == "process_refund" and tool_input.get("amount", 0) > 500:
        return False
    return True

### Step 2: Simulate the agentic loop enforcing the gate

In [ ]:
def handle_request(customer_name: str, refund_amount: float):
    customer_id = get_customer(customer_name)
    if not pre_tool_call_hook("process_refund", {"amount": refund_amount}, customer_id):
        return "BLOCKED: escalate to human (unverified identity or amount over threshold)"
    return process_refund(customer_id, refund_amount)

print(handle_request("Jane Doe", 120))
print(handle_request("Unknown Person", 120))
print(handle_request("Jane Doe", 900))

### Step 3: Decompose a multi-issue request

In [ ]:
multi_issue_request = "I want a refund for order #55, AND my account email is wrong, AND I want a status update on order #61."
sub_issues = decompose_code_review.__doc__ and [
    "Investigate refund for order #55",
    "Correct account email",
    "Provide status update on order #61",
]
print(sub_issues)

## Lab Exercises

### Exercise 1: Stop-reason loop correctness

**Objective**: Fix a buggy agentic loop that checks assistant text content instead of stop_reason.

**Produce**: A corrected loop function and a test showing it terminates correctly.

In [ ]:
class FakeResponse:
    def __init__(self, stop_reason, content):
        self.stop_reason = stop_reason
        self.content = content

def buggy_loop_check(response) -> bool:
    # BUG: checks for text content instead of stop_reason - anti-pattern!
    return any(getattr(b, "type", None) == "text" for b in response.content)

# TODO: write a corrected version that checks response.stop_reason == "end_turn" instead
def fixed_loop_check(response) -> bool:
    pass

r = FakeResponse("tool_use", [])
print(fixed_loop_check(r))  # should be False - still needs to call tools

# Expected output: fixed_loop_check returns False for stop_reason='tool_use' and True for 'end_turn'.

In [ ]:
# STRETCH GOAL: write a test harness that runs both buggy_loop_check and fixed_loop_check
# against 3 FakeResponse cases and reports where they disagree.
cases = [FakeResponse("tool_use", []), FakeResponse("end_turn", []), FakeResponse("tool_use", [])]
# TODO: compare outputs of both functions across all cases

# Expected output: A report showing the buggy version can misclassify at least one case relative to the fixed version.

### Exercise 2: Dynamic subagent selection

**Objective**: Extend coordinator_decide_subagents to avoid invoking unnecessary subagents for narrow queries.

**Produce**: An updated function and 3 test queries showing different subagent sets.

In [ ]:
def coordinator_decide_subagents_v2(query: str) -> list:
    # TODO: only include "web_search_agent" if the query needs current/external info,
    # only include "document_analysis_agent" if a document is referenced, always include synthesis
    pass

print(coordinator_decide_subagents_v2("Summarize the attached report."))
print(coordinator_decide_subagents_v2("What's trending in AI safety research right now?"))
print(coordinator_decide_subagents_v2("What's trending, and how does it relate to the attached report?"))

# Expected output: The 3 test queries produce 3 different, appropriately-scoped subagent lists.

In [ ]:
# STRETCH GOAL: simulate parallel invocation by returning subagent calls as a single batch
# (list) rather than issuing them one at a time, and print the "latency win" vs sequential.
import time
def simulate_latency(subagents, parallel: bool) -> float:
    # TODO: return sum(1 for _ in subagents) if not parallel else max(1 for _ in subagents) (simplified units)
    pass
print(simulate_latency(["a","b","c"], parallel=False), simulate_latency(["a","b","c"], parallel=True))

# Expected output: Parallel simulated latency is lower than sequential for the same subagent set.

### Exercise 3: Business-rule hook

**Objective**: Add a second business rule to pre_tool_call_hook (e.g., block deletes without manager approval).

**Produce**: An updated hook function and 3 test cases.

In [ ]:
def pre_tool_call_hook_v2(tool_name, tool_input, verified_customer_id, manager_approved=False):
    if tool_name == "process_refund" and not verified_customer_id:
        return False
    if tool_name == "process_refund" and tool_input.get("amount", 0) > 500:
        return False
    # TODO: also block "delete_account" unless manager_approved is True
    return True

print(pre_tool_call_hook_v2("delete_account", {}, "C123", manager_approved=False))
print(pre_tool_call_hook_v2("delete_account", {}, "C123", manager_approved=True))

# Expected output: delete_account is blocked without manager approval, allowed with it.

In [ ]:
# STRETCH GOAL: log every blocked call with a reason string instead of just returning False.
blocked_log = []
def pre_tool_call_hook_v3(tool_name, tool_input, verified_customer_id, manager_approved=False):
    # TODO: append a (tool_name, reason) tuple to blocked_log whenever blocking, then return False
    pass
pre_tool_call_hook_v3("delete_account", {}, "C123", manager_approved=False)
print(blocked_log)

# Expected output: blocked_log contains an entry explaining why the delete_account call was blocked.

### Exercise 4: Adaptive vs. chained decomposition

**Objective**: Given a task description, decide whether it needs prompt chaining or adaptive decomposition, and implement accordingly.

**Produce**: A decision plus an implemented decomposition for one open-ended task.

In [ ]:
task = "Add comprehensive tests to a legacy codebase with unknown structure."

# TODO: decide: is this predictable (prompt chaining) or open-ended (adaptive decomposition)?
decision = "..."  # TODO

# TODO: implement an adaptive plan: first map structure, then identify high-impact areas, then a prioritized plan
adaptive_plan = []  # TODO: list of ordered sub-steps
print(decision)
print(adaptive_plan)

# Expected output: Correctly identifies this as needing adaptive decomposition, with a sensible 3+ step plan.

In [ ]:
# STRETCH GOAL: simulate the plan adapting after step 1 reveals new information
# (e.g., discovering an untested payments module raises its priority).
discovered_info = "Found an untested payments module with no test coverage at all."
# TODO: update adaptive_plan based on discovered_info
updated_plan = adaptive_plan  # TODO
print(updated_plan)

# Expected output: The updated plan reflects the newly discovered high-priority module.

### Exercise 5: Session resumption vs. fresh start

**Objective**: Decide session strategy for 3 scenarios using choose_session_strategy.

**Produce**: 3 scenario -> strategy mappings.

In [ ]:
scenarios = [
    ("Resuming tomorrow to continue the same investigation, nothing changed", True, False),
    ("Comparing 2 different refactor approaches from the same analysis", True, True),
    ("Resuming after major unrelated code changes since yesterday", False, False),
]
# TODO: call choose_session_strategy(prior_context_valid, need_parallel_exploration) for each and print results
for desc, valid, parallel in scenarios:
    pass

# Expected output: Correct strategy chosen for each of the 3 scenarios per the concept explanation above.

In [ ]:
# STRETCH GOAL: write the actual structured summary text you would inject for the "start fresh" case.
fresh_start_summary = "..."  # TODO
print(fresh_start_summary)

# Expected output: A concise, structured summary covering the key facts a fresh session would need.

## Self-Assessment: Agentic Architecture & Orchestration

Answer each question, then check the Answers cell below.


**Q1.** What are the two `stop_reason` values that determine whether an agentic loop continues or terminates?

**Q2.** Why is it an anti-pattern to check for assistant text content as a completion indicator?

**Q3.** Why don't subagents automatically inherit the coordinator's full conversation history?

**Q4.** When must you use a hook rather than a prompt instruction to enforce a business rule?

**Q5.** When is `fork_session` the right choice instead of `--resume`?

## Answers


**A1.** "tool_use" (continue: execute tools and loop again) and "end_turn" (stop: present the final response).

**A2.** Because text content can appear alongside tool calls or vary in ways unrelated to actual completion, making it an unreliable signal.

**A3.** Subagents operate with isolated context by design; relevant findings must be explicitly included in their prompts.

**A4.** When compliance must be deterministic/guaranteed (e.g., verifying identity before a refund) — prompt-based compliance is only probabilistic.

**A5.** When you need to explore multiple divergent approaches in parallel from a shared analysis baseline, rather than continuing one linear conversation.

---
**Next**: [Domain 2: Tool Design & MCP Integration](./02-tool-design-and-mcp-integration.ipynb)
**Back**: [README](./README.md)